In [1]:
# Install evaluation metric libraries:
# rouge-score — measures text overlap between model answer and correct answer
#               good for checking if key words and phrases match
# bert-score  — measures semantic similarity using BERT embeddings
#               smarter than ROUGE — understands meaning, not just exact word matches

!pip install rouge-score bert-score -q

print("All libraries installed!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00
All libraries installed!


In [5]:
import pandas as pd
from google.colab import files

# Upload the correct answers dataset
print("Upload Austrian_Tax_Law_Dataset_-_Dataset.csv...")
files.upload()
correct_df = pd.read_csv("Austrian Tax Law Dataset - Dataset.csv")

# Keep only id and correct_answer columns
correct_df = correct_df[["id", "correct_answer"]].dropna(subset=["correct_answer"])
print(f"Correct answers loaded: {len(correct_df)}")

# Upload all 3 model result files
print("\nUpload model1_mistral_results.csv...")
files.upload()
model1_df = pd.read_csv("model1_mistral_results.csv")

print("Upload model2_finetuned_results.csv...")
files.upload()
model2_df = pd.read_csv("model2_finetuned_results.csv")

print("Upload model3_rag_results.csv...")
files.upload()
model3_df = pd.read_csv("model3_rag_results.csv")

print(f"\nModel 1 answers: {len(model1_df)}")
print(f"Model 2 answers: {len(model2_df)}")
print(f"Model 3 answers: {len(model3_df)}")

Upload Austrian_Tax_Law_Dataset_-_Dataset.csv...


Saving Austrian Tax Law Dataset - Dataset.csv to Austrian Tax Law Dataset - Dataset (3).csv
Correct answers loaded: 645

Upload model1_mistral_results.csv...


Saving model1_mistral_results.csv to model1_mistral_results.csv
Upload model2_finetuned_results.csv...


Saving model2_finetuned_results.csv to model2_finetuned_results.csv
Upload model3_rag_results.csv...


Saving model3_rag_results.csv to model3_rag_results.csv

Model 1 answers: 643
Model 2 answers: 643
Model 3 answers: 643


In [6]:
# Merge each models answers with the correct answers on the question ID This
# ensures we only evaluate questions,
# that have both a model answer and a correct answer

model1_merged = model1_df.merge(correct_df, on="id")
model2_merged = model2_df.merge(correct_df, on="id")
model3_merged = model3_df.merge(correct_df, on="id")

print(f"Model 1 questions to evaluate: {len(model1_merged)}")
print(f"Model 2 questions to evaluate: {len(model2_merged)}")
print(f"Model 3 questions to evaluate: {len(model3_merged)}")

# Quick sanity check, show one example
print(f"\nExample question: {model1_merged['id'].iloc[0]}")
print(f"Correct answer:   {model1_merged['correct_answer'].iloc[0][:100]}...")
print(f"Model 1 answer:   {model1_merged['answer'].iloc[0][:100]}...")

Model 1 questions to evaluate: 643
Model 2 questions to evaluate: 643
Model 3 questions to evaluate: 643

Example question: CORP-TAX-001
Correct answer:   Das Einkommen, das der unbeschränkt Steuerpflichtige innerhalb eines Kalenderjahres bezogen hat (§ 7...
Model 1 answer:   Die Steuerbemessungsgrundlage für die Körperschaftsteuer in Österreich ist der Gewinn des Unternehme...


In [7]:
from rouge_score import rouge_scorer

# Initialize ROUGE scorer
# rouge1 — measures overlap of individual words
# rougeL — measures longest common subsequence (word order matters)
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

def calculate_rouge(merged_df):
    rouge1_scores = []
    rougeL_scores = []

    for _, row in merged_df.iterrows():
        # Compare model answer against correct answer
        scores = scorer.score(
            str(row['correct_answer']),  # reference (correct)
            str(row['answer'])           # hypothesis (model output)
        )
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

    return {
        'ROUGE-1': round(sum(rouge1_scores) / len(rouge1_scores), 4),
        'ROUGE-L': round(sum(rougeL_scores) / len(rougeL_scores), 4)
    }

# Calculate ROUGE for all 3 models
print("Calculating ROUGE scores...")
rouge_model1 = calculate_rouge(model1_merged)
rouge_model2 = calculate_rouge(model2_merged)
rouge_model3 = calculate_rouge(model3_merged)

print(f"Model 1 (Inference): {rouge_model1}")
print(f"Model 2 (Fine-tune): {rouge_model2}")
print(f"Model 3 (RAG):       {rouge_model3}")

Calculating ROUGE scores...
Model 1 (Inference): {'ROUGE-1': 0.2073, 'ROUGE-L': 0.1379}
Model 2 (Fine-tune): {'ROUGE-1': 0.226, 'ROUGE-L': 0.1649}
Model 3 (RAG):       {'ROUGE-1': 0.194, 'ROUGE-L': 0.1317}


In [8]:
from bert_score import score as bert_score

def calculate_bertscore(merged_df):
    references = merged_df['correct_answer'].astype(str).tolist()
    hypotheses = merged_df['answer'].astype(str).tolist()

    # Calculate BERTScore — lang='de' because our answers are in German
    # This uses a multilingual BERT model to measure semantic similarity
    P, R, F1 = bert_score(hypotheses, references, lang='de', verbose=False)

    return {
        'BERTScore-P': round(P.mean().item(), 4),
        'BERTScore-R': round(R.mean().item(), 4),
        'BERTScore-F1': round(F1.mean().item(), 4)
    }

print("Calculating BERTScore (this takes a few minutes)...")
bert_model1 = calculate_bertscore(model1_merged)
bert_model2 = calculate_bertscore(model2_merged)
bert_model3 = calculate_bertscore(model3_merged)

print(f"Model 1 (Inference): {bert_model1}")
print(f"Model 2 (Fine-tune): {bert_model2}")
print(f"Model 3 (RAG):       {bert_model3}")

Calculating BERTScore (this takes a few minutes)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model 1 (Inference): {'BERTScore-P': 0.6733, 'BERTScore-R': 0.7119, 'BERTScore-F1': 0.6913}
Model 2 (Fine-tune): {'BERTScore-P': 0.7283, 'BERTScore-R': 0.7146, 'BERTScore-F1': 0.7204}
Model 3 (RAG):       {'BERTScore-P': 0.653, 'BERTScore-R': 0.7196, 'BERTScore-F1': 0.6841}


In [2]:
import pandas as pd

# Results from our evaluation runs
results = pd.DataFrame({
    'Model': ['Model 1 - Inference (Mistral 7B)', 'Model 2 - Fine-tuning (QLoRA)', 'Model 3 - RAG (Groq Llama)'],
    'ROUGE-1': [0.2073, 0.226, 0.194],
    'ROUGE-L': [0.1379, 0.1649, 0.1317],
    'BERTScore-P': [0.6733, 0.7283, 0.653],
    'BERTScore-R': [0.7119, 0.7146, 0.7196],
    'BERTScore-F1': [0.6913, 0.7204, 0.6841],
})

print("MAIN RESULTS TABLE")
print(results.to_string(index=False))

MAIN RESULTS TABLE
                           Model  ROUGE-1  ROUGE-L  BERTScore-P  BERTScore-R  BERTScore-F1
Model 1 - Inference (Mistral 7B)   0.2073   0.1379       0.6733       0.7119        0.6913
   Model 2 - Fine-tuning (QLoRA)   0.2260   0.1649       0.7283       0.7146        0.7204
      Model 3 - RAG (Groq Llama)   0.1940   0.1317       0.6530       0.7196        0.6841


In [3]:
# Save results table to CSV for the project report
results.to_csv("evaluation_results.csv", index=False)
print("Results saved!")

from google.colab import files
files.download("evaluation_results.csv")

Results saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>